In [2]:
import sqlite3
import numpy
import pandas
import plotly.express
import plotly.graph_objects
import dash
import dash.dcc
import dash.html
import dash.dependencies

# Connect to the SQLite database
db_path = '/home/dimitri/code/oll_onemax/computed/fire/_merged.db'
conn = sqlite3.connect(db_path)

# Load table names
query_tables = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pandas.read_sql_query(query_tables, conn)

# Assuming table names and columns as placeholders
evaluation_table = 'EVALUATION_EPISODES'  # Replace with actual table name
policies_table = 'CONSTRUCTED_POLICIES'  # Replace with actual table name

# Column to display
column_to_display = 'num_function_evaluations'  # Replace with actual column name if different

# Check if the table exists and the column is correct
assert evaluation_table in tables['name'].to_list(), f"Table '{evaluation_table}' does not exist in the database."

# Load data from the evaluation table
query_evaluation = f"""
  SELECT policy_id,
         AVG({column_to_display}) AS avg_value,
         COUNT(*) AS row_count,
         AVG(({column_to_display} - avg_value) * ({column_to_display} - avg_value)) AS var_value
  FROM (
      SELECT policy_id,
             {column_to_display},
             AVG({column_to_display}) OVER (PARTITION BY policy_id) AS avg_value
      FROM {evaluation_table}
  )
  GROUP BY policy_id
"""
df_evaluation = pandas.read_sql_query(query_evaluation, conn)
df_evaluation.set_index('policy_id', inplace=True)

# Load num_total_timesteps from the policies table
assert policies_table in tables['name'].to_list(), f"Table '{policies_table}' does not exist in the database."
query_policies = f"""
  SELECT policy_id, num_total_timesteps, db_path
  FROM {policies_table}
"""
df_policies = pandas.read_sql_query(query_policies, conn)

# Merge the two dataframes on policy_id
df_merged = df_evaluation.merge(df_policies.drop_duplicates(subset='policy_id'), on='policy_id', how='left')
df_merged.set_index('policy_id', inplace=True)

# Calculate standard deviation
df_merged['stddev_value'] = numpy.sqrt(df_merged['var_value'])

# Separate the baseline data (policy_id = -1)
df_baseline = df_merged[df_merged.index == -1]
df_others = df_merged[df_merged.index != -1]

# Get the number of unique policies
num_policies = df_policies['db_path'].nunique()

# Add number of episodes per policy
df_others.loc[:, 'episodes_per_policy'] = df_others['row_count'] / num_policies

# Create the line plot for other policies
fig = plotly.express.line(
  df_others,
  x='num_total_timesteps',
  y='avg_value',
  labels={'num_total_timesteps': 'Number of Total Timesteps', 'avg_value': f'Average {column_to_display.replace("_", " ").title()}'},
  title=f'Average {column_to_display.replace("_", " ").title()} by Number of Total Timesteps'
)

# Update the plot with hover data for other policies
fig.update_traces(
  mode='markers+lines',
  hovertemplate='<b>Number of Total Timesteps:</b> %{x}<br>' +
                '<b>Average Value:</b> %{y}<br>' +
                '<b>Row Count:</b> %{customdata[0]}<br>' +
                '<b>Number of Policies:</b> ' + str(num_policies) + '<br>' +
                '<b>Number of Episodes per Policy:</b> %{customdata[1]}<br>' +
                '<b>Standard Deviation:</b> %{customdata[2]}'
)

# Add hover data for other policies
fig.update_traces(customdata=df_others[['row_count', 'episodes_per_policy', 'stddev_value']])

# Add the standard deviation shading for other policies
fig.add_traces([
  plotly_graph_objects.Scatter(
    x=df_others['num_total_timesteps'],
    y=df_others['avg_value'] + df_others['stddev_value'],
    mode='lines',
    line=dict(width=0),
    showlegend=False,
    hoverinfo='skip'
  ),
  plotly_graph_objects.Scatter(
    x=df_others['num_total_timesteps'],
    y=df_others['avg_value'] - df_others['stddev_value'],
    mode='lines',
    line=dict(width=0),
    fill='tonexty',
    fillcolor='rgba(0,100,80,0.2)',
    showlegend=False,
    hoverinfo='skip'
  )
])

# Add the baseline line with hoverable points and standard deviation shading
if not df_baseline.empty:
  baseline_value = df_baseline['avg_value'].values[0]
  baseline_stddev = numpy.sqrt(df_baseline['var_value'].values[0])

  fig.add_trace(
    plotly_graph_objects.Scatter(
      x=df_others['num_total_timesteps'],
      y=[baseline_value] * len(df_others),
      mode='lines',
      line=dict(color='orange', dash='dash'),
      name='Baseline',
      hoverinfo='y',
      hovertemplate='<b>Baseline:</b><br>' +
                    '<b>Average Value:</b> %{y}<br>' +
                    f'<b>Standard Deviation:</b> {baseline_stddev}'
    )
  )

  fig.add_traces([
    plotly_graph_objects.Scatter(
      x=df_others['num_total_timesteps'],
      y=[baseline_value + baseline_stddev] * len(df_others),
      mode='lines',
      line=dict(width=0),
      showlegend=False,
      hoverinfo='skip'
    ),
    plotly_graph_objects.Scatter(
      x=df_others['num_total_timesteps'],
      y=[baseline_value - baseline_stddev] * len(df_others),
      mode='lines',
      line=dict(width=0),
      fill='tonexty',
      fillcolor='rgba(255,165,0,0.2)',
      showlegend=False,
      hoverinfo='skip'
    )
  ])

# Update layout to start y-axis from 0 and set dragmode to select
fig.update_layout(
  yaxis=dict(range=[0, None]),
  dragmode='select'
)

# Create the Dash app
app = dash.Dash(__name__)

app.layout = dash.html.Div([
  dash.dcc.Graph(id='line-plot', figure=fig, config={'scrollZoom': True}),
  dash.html.Div(id='selected-data'),
  dash.dcc.Graph(id='horizontal-lines-plot')
])

@app.callback(
  [dash.dependencies.Output('selected-data', 'children'),
   dash.dependencies.Output('horizontal-lines-plot', 'figure')],
  [dash.dependencies.Input('line-plot', 'selectedData')]
)
def display_selected_data(selectedData):
  if selectedData is None:
    return "No points selected", plotly_graph_objects.Figure()
  points = selectedData['points']
  selected_points = [
    {
      'x': point['x'],
      'y': point['y'],
      'customdata': point['customdata']
    }
    for point in points
  ]

  # Create a secondary plot with horizontal lines at the "Number of Total Timesteps" of each selected point
  lines_fig = plotly_graph_objects.Figure()
  for point in points:
    num_total_timesteps = point['x']
    lines_fig.add_trace(
      plotly_graph_objects.Scatter(
        x=[0, max(df_others['num_total_timesteps'])],
        y=[num_total_timesteps, num_total_timesteps],
        mode='lines',
        line=dict(color='blue', dash='dash'),
        showlegend=False,
        hovertemplate='<b>Number of Total Timesteps:</b> %{y}'
      )
    )

  lines_fig.update_layout(
    title='Horizontal Lines Plot',
    xaxis_title='Number of Total Timesteps',
    yaxis_title='Number of Total Timesteps'
  )

  return dash.html.Pre(str(selected_points)), lines_fig

# Run the app
if __name__ == '__main__':
  app.run_server(debug=True, use_reloader=False)
  conn.close()


/tmp/ipykernel_3108608/2731325528.py:7: UserWarning:


The dash_core_components package is deprecated. Please replace
`import dash_core_components as dcc` with `from dash import dcc`

/tmp/ipykernel_3108608/2731325528.py:8: UserWarning:


The dash_html_components package is deprecated. Please replace
`import dash_html_components as html` with `from dash import html`

/tmp/ipykernel_3108608/2731325528.py:69: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



NameError: name 'plotly_graph_objects' is not defined